# Part 2 — Exercise solutions

In [ ]:
import os

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../../.env", override=True)
client = genai.Client()
EMBED_MODEL = "gemini-embedding-001"


def embed(texts, dim=768):
    if isinstance(texts, str):
        texts = [texts]
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", output_dimensionality=dim),
    )
    E = np.array([e.values for e in result.embeddings])
    return E / np.linalg.norm(E, axis=1, keepdims=True)

## 5.1 — Odd one out

In [ ]:
def odd_one_out(sentences):
    E = embed(sentences)
    S = E @ E.T
    # average similarity to the *others*: drop self-similarity (the diagonal 1s)
    avg = (S.sum(axis=1) - 1) / (len(sentences) - 1)
    return sentences[int(avg.argmin())]


sentences = [
    "The boss fight was brutally difficult but fair.",
    "I couldn't beat the final boss, way too hard.",
    "The soundtrack is absolutely gorgeous.",
    "My sourdough bread came out perfect today.",
]
print(odd_one_out(sentences))

In [ ]:
sneaky = [
    "The bank raised interest rates again.",
    "I sat on the river bank watching the water.",   # same word, different meaning
    "Mortgage payments are getting expensive.",
    "The central bank announced new policy.",
]
print(odd_one_out(sneaky))  # embeddings read context, not spelling

## 5.2 — Vectorized cosine matrix

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def cosine_matrix(A):
    N = A / np.linalg.norm(A, axis=1, keepdims=True)
    return N @ N.T


A = np.random.default_rng(1).random((6, 10)) - 0.5
loops = np.array([[cosine(A[i], A[j]) for j in range(len(A))] for i in range(len(A))])
print("match:", np.allclose(cosine_matrix(A), loops))

## 5.3 — Your languages

In [ ]:
trio = [
    "The last level was impossibly hard.",
    "Ultimul nivel a fost imposibil de greu.",
    "I ordered a pizza with extra cheese.",
]
E = embed(trio)
S = E @ E.T
print(f"EN↔RO (same meaning):      {S[0, 1]:.3f}")
print(f"EN↔pizza (same language):  {S[0, 2]:.3f}")

## 5.4 — Shrinking dimensions

In [ ]:
sentences = [
    "The boss fight was brutally difficult but fair.",
    "I couldn't beat the final boss, way too hard.",
    "The soundtrack is absolutely gorgeous.",
    "The game crashes every time I open the map.",
    "Constant freezes whenever I fast-travel.",
    "My sourdough bread came out perfect today.",
]

for dim in [768, 128]:
    E = embed(sentences, dim=dim)
    sims = (E @ E.T)[0]                      # similarities vs sentence 0
    order = np.argsort(sims)[::-1]
    print(f"dim {dim:>4}: ranking vs sentence 0 → {list(order)}")

The *values* shift slightly, but the *ranking* barely moves — Gemini embeddings
are trained (Matryoshka-style) so truncated vectors still work. Smaller = cheaper
storage and faster search, at a small quality cost.